# HW2 — Module 2: Combining and processing data from multiple sources

**Goal:** Answer the four Module 2 homework questions: withdrawn IPO value by company
type, median Sharpe ratio for 2025 IPOs, optimal fixed holding period for IPO stocks,
and profit from an RSI<30 entry rule.

**Inputs:**
- iposcoop.com recently-filed IPO table (live HTML)
- iposcoop.com 2025 pricings table (live HTML)
- yfinance daily OHLCV for the 2025 IPO tickers
- Course-provided parquet of precomputed technical/macro indicators (Google Drive)

**Outputs:**
- Four numeric answers for the HW2 submission form
- No persisted data artifacts

In [1]:
import numpy as np
import pandas as pd

url = 'https://www.iposcoop.com/ipos-recently-filed/'
tables = pd.read_html(url)

print(f'tables returned: {len(tables)}')
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns))

tables returned: 1
0 (500, 10) ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']


## Q1: Withdrawn IPOs by company type

The recently-filed table loads as a single 500-row table. Next we isolate the
withdrawn filings. The assignment expects 32 rows; the page is live and we're
reading it on Sep 14 rather than Sep 11, so we check the count rather than
assume it.

In [2]:
ipos_filed = tables[0]

print(ipos_filed['Expected To Trade'].value_counts(dropna=False))

withdrawn = ipos_filed[ipos_filed['Expected To Trade'] == 'Withdrawn'].copy()
print(f'\nwithdrawn rows: {len(withdrawn)}')
withdrawn.head()

Expected To Trade
Priced       272
TBA          175
Withdrawn     32
Week of        8
Friday         4
Wednesday      3
Tuesday        2
Thursday       2
Monday         1
Postponed      1
Name: count, dtype: int64

withdrawn rows: 32


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
7,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
13,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O
22,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
24,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.0,$4.00,$5.00,$22.50,Withdrawn,S/O
32,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.0,$10.00,$10.00,$200.00,Withdrawn,S/O


### Company Type

Assign each withdrawn filing to a company class by matching patterns in the
company name. Order matters: the first pattern that matches wins, so
"Acquisition Corp" is tested before "Inc" and "Corp" before "Holdings".

In [3]:
def classify_company(name: str) -> str:
    if 'Technologies' in name:
        return 'Technologies'
    if 'Acquisition Corp' in name or 'Acquisition Corporation' in name or 'Corp' in name:
        return 'Acquisition Corp'
    if 'Inc' in name or 'Incorporated' in name:
        return 'Inc.'
    if 'Group' in name:
        return 'Group'
    if 'Ltd' in name or 'Limited' in name:
        return 'Limited'
    if 'Holdings' in name or 'Holding' in name:
        return 'Holdings'
    return 'Other'

withdrawn['Company Type'] = withdrawn['Company'].apply(classify_company)

print(withdrawn['Company Type'].value_counts())
withdrawn[['Company', 'Company Type']]

Company Type
Other               8
Limited             8
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64


,Company,Company Type
7,Motive Technologies (Withdrawn),Technologies
13,Idea Tech Holding (Withdrawn),Holdings
22,Hornbeck Offshore Services (Withdrawn),Other
24,Coolbit Technologies Ltd. (Withdrawn),Technologies
32,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp
45,pan-Africa Corp. (Withdrawn),Acquisition Corp
61,Living Homeopathy (Withdrawn),Other
105,Nuclea Energy (Withdrawn),Other
136,APEX Global Solutions Ltd. (WITHDRAWN),Limited
141,Cloud Data Holdings (Withdrawn),Holdings


### Price parsing and numeric conversion

'Price Low' and 'Price High' arrive as currency strings ('$4.00') or NaN; the
average of the two becomes Avg_price. 'Est $ Vol (millions)' is also a currency
string. 'Shares (millions)' already loaded as float, but we run it through the
same cleaner so the code doesn't depend on read_html's type inference.

In [4]:
def parse_price(value):
    """'$8.00' -> 8.0; '-', NaN, or unparseable -> NaN."""
    if pd.isna(value):
        return np.nan
    cleaned = str(value).replace('$', '').replace(',', '').strip()
    return pd.to_numeric(cleaned, errors='coerce')

withdrawn['Price Low Num'] = withdrawn['Price Low'].apply(parse_price)
withdrawn['Price High Num'] = withdrawn['Price High'].apply(parse_price)
withdrawn['Avg_price'] = withdrawn[['Price Low Num', 'Price High Num']].mean(axis=1)

withdrawn['Shares (millions)'] = withdrawn['Shares (millions)'].apply(parse_price)
withdrawn['Est $ Vol (millions)'] = withdrawn['Est $ Vol (millions)'].apply(parse_price)

print(withdrawn[['Shares (millions)', 'Avg_price', 'Est $ Vol (millions)']].isnull().sum())
print()
print('shares == 0 with a price present:',
      ((withdrawn['Shares (millions)'] == 0) & withdrawn['Avg_price'].notna()).sum())

withdrawn

Shares (millions)       0
Avg_price               6
Est $ Vol (millions)    0
dtype: int64

shares == 0 with a price present: 0


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Price Low Num,Price High Num,Avg_price
7,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Technologies,NaN,NaN,NaN
13,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,9.00,Withdrawn,S/O,Holdings,4.0,5.0,4.50
22,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,100.00,Withdrawn,S/O,Other,NaN,NaN,NaN
24,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,22.50,Withdrawn,S/O,Technologies,4.0,5.0,4.50
32,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,200.00,Withdrawn,S/O,Acquisition Corp,10.0,10.0,10.00
45,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,125.00,Withdrawn,S/O,Acquisition Corp,10.0,10.0,10.00
61,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,19.00,Withdrawn,S/O,Other,4.0,6.0,5.00
105,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,50.40,Withdrawn,S/O,Other,8.0,10.0,9.00
136,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,15.00,Withdrawn,S/O,Limited,4.0,4.0,4.00
141,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,15.94,Withdrawn,S/O,Holdings,4.0,4.5,4.25


### Shares_offered_value

Prefer Shares (millions) x Avg_price; fall back to the site's own
Est $ Vol (millions) where the product is null. The six rows with no
disclosed price range are the ones that take the fallback.

In [5]:
product = withdrawn['Shares (millions)'] * withdrawn['Avg_price']
withdrawn['Shares_offered_value'] = product.fillna(withdrawn['Est $ Vol (millions)'])

print('rows using the Est $ Vol fallback:', product.isna().sum())
print('nulls remaining in Shares_offered_value:',
      withdrawn['Shares_offered_value'].isna().sum())
print('total value ($M):', withdrawn['Shares_offered_value'].sum().round(2))

withdrawn[['Company', 'Company Type', 'Shares (millions)', 'Avg_price',
           'Est $ Vol (millions)', 'Shares_offered_value']]

rows using the Est $ Vol fallback: 6
nulls remaining in Shares_offered_value: 0
total value ($M): 1874.34


,Company,Company Type,Shares (millions),Avg_price,Est $ Vol (millions),Shares_offered_value
7,Motive Technologies (Withdrawn),Technologies,0.00,NaN,100.00,100.0000
13,Idea Tech Holding (Withdrawn),Holdings,2.00,4.50,9.00,9.0000
22,Hornbeck Offshore Services (Withdrawn),Other,0.00,NaN,100.00,100.0000
24,Coolbit Technologies Ltd. (Withdrawn),Technologies,5.00,4.50,22.50,22.5000
32,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.00,10.00,200.00,200.0000
45,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.50,10.00,125.00,125.0000
61,Living Homeopathy (Withdrawn),Other,3.75,5.00,19.00,18.7500
105,Nuclea Energy (Withdrawn),Other,5.60,9.00,50.40,50.4000
136,APEX Global Solutions Ltd. (WITHDRAWN),Limited,3.75,4.00,15.00,15.0000
141,Cloud Data Holdings (Withdrawn),Holdings,3.75,4.25,15.94,15.9375


### Aggregation by company type

Sum Shares_offered_value within each class. The class totals must add back
to the $1,874.34M computed above.

In [6]:
by_type = (withdrawn
           .groupby('Company Type')['Shares_offered_value']
           .agg(['sum', 'count'])
           .sort_values('sum', ascending=False))

print(by_type.round(2))
print()
print('sum of class totals:', by_type['sum'].sum().round(2))

                     sum  count
Company Type                   
Acquisition Corp  499.98      5
Inc.              351.00      1
Holdings          311.66      4
Other             290.44      8
Limited           203.85      8
Technologies      184.90      3
Group              32.50      3

sum of class totals: 1874.34


**Q1 answer:** Acquisition Corp, with $499.98M in total withdrawn offering value
across 5 filings.

Composition matters for interpreting this: the Acquisition Corp class is
predominantly SPACs, which file at a standard $10.00 price with round share
counts, so the class total reflects the count of SPAC withdrawals more than any
per-deal size. The runner-up class, Inc. at $351.00M, is a single filing
(Clear Street Group).

## Q2: Median Sharpe ratio for 2025 IPOs

Load the 2025 pricings table (231 IPOs expected), then filter to offerings
before 1 September 2025, excluding rows with a 0% return. The assignment
expects 148 tickers to survive. As with Q1 this is a live page, so we check
the counts rather than assume them.

In [7]:
ipos_2025_url = 'https://www.iposcoop.com/2025-pricings/'
tables_2025 = pd.read_html(ipos_2025_url)

print(f'tables returned: {len(tables_2025)}')
for i, t in enumerate(tables_2025):
    print(i, t.shape, list(t.columns))

ipos_2025 = tables_2025[0]
ipos_2025.head()

tables returned: 1
0 (231, 10) ['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


### Inspect dtypes and clean the numeric columns

read_html's type inference is per-column and not guaranteed, so check what
actually loaded before relying on it. Offer Date and Return drive the filter;
the price columns aren't required downstream but 1st Day Close serves as an
independent check on the Q3 entry price.

In [8]:
print(ipos_2025.dtypes)
print()

ipos_2025['Offer Date'] = pd.to_datetime(ipos_2025['Offer Date'], format='mixed')
ipos_2025['Return'] = pd.to_numeric(
    ipos_2025['Return'].astype(str).str.replace('%', '', regex=False),
    errors='coerce') / 100

for col in ['Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price']:
    ipos_2025[col] = ipos_2025[col].apply(parse_price)

print(ipos_2025.dtypes)
print()
print(ipos_2025[['Shares (millions)', 'Offer Price', '1st Day Close',
                 'Current Price', 'Return']].isnull().sum())

Company                  str
Symbol                   str
Industry                 str
Offer Date               str
Shares (millions)    float64
Offer Price              str
1st Day Close            str
Current Price            str
Return                   str
SCOOP Rating             str
dtype: object

Company                         str
Symbol                          str
Industry                        str
Offer Date           datetime64[us]
Shares (millions)           float64
Offer Price                 float64
1st Day Close               float64
Current Price               float64
Return                      float64
SCOOP Rating                    str
dtype: object

Shares (millions)    0
Offer Price          0
1st Day Close        0
Current Price        0
Return               0
dtype: int64


### Filter to pre-September 2025 offerings

Keep offerings before 2025-09-01 with a non-zero return. Rows with a 0% return
are placeholders (blank-check SPACs and similar with $0.00 prices) rather than
tradeable history. The assignment expects 148 tickers.

In [9]:
print('Offer Date range:', ipos_2025['Offer Date'].min().date(),
      'to', ipos_2025['Offer Date'].max().date())
print('null returns:', ipos_2025['Return'].isna().sum())
print('zero returns:', (ipos_2025['Return'] == 0).sum())

filtered_ipos = ipos_2025[
    (ipos_2025['Offer Date'] < '2025-09-01') &
    (ipos_2025['Return'] != 0)
].copy()

print(f'\nfiltered rows: {len(filtered_ipos)}')
print('unique symbols:', filtered_ipos['Symbol'].nunique())
filtered_ipos.head()

Offer Date range: 2025-01-02 to 2025-12-18
null returns: 0
zero returns: 27

filtered rows: 146
unique symbols: 146


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,4.0,4.59,1.68,-0.5800,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,4.0,3.38,0.40,-0.9000,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,4.0,6.02,29.47,6.3675,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,4.0,4.02,0.37,-0.9075,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,10.0,10.12,10.12,0.0120,S/O


### Reconciling 146 against the expected 148

The assignment expects 148 tickers. Two filters are in play: the date cut and
the non-zero return test. Isolate each to see which one accounts for the
difference, and inspect the zero-return rows in the pre-September window to
judge whether they are placeholders or real listings sitting at exactly 0%.

In [10]:
pre_sept = ipos_2025[ipos_2025['Offer Date'] < '2025-09-01']
print('rows before 2025-09-01 (date filter only):', len(pre_sept))
print('of those, zero return:', (pre_sept['Return'] == 0).sum())
print('offer dates in the 08-29 to 09-02 window:')
print(ipos_2025.loc[ipos_2025['Offer Date'].between('2025-08-29', '2025-09-02'),
                    ['Company', 'Symbol', 'Offer Date', 'Return']])

pre_sept[pre_sept['Return'] == 0][
    ['Company', 'Symbol', 'Offer Date', 'Offer Price',
     '1st Day Close', 'Current Price', 'Return']]

rows before 2025-09-01 (date filter only): 163
of those, zero return: 17
offer dates in the 08-29 to 09-02 window:
                 Company Symbol Offer Date  Return
68  Picard Medical, Inc.    PMI 2025-08-29   -0.58


,Company,Symbol,Offer Date,Offer Price,1st Day Close,Current Price,Return
103,"Robot Consulting Co., Ltd.",LAWR,2025-07-17,4.00,0.00,0.00,0.0
104,Aura Minerals (NASDAQ Uplisting),AUGO,2025-07-16,24.25,24.25,24.25,0.0
108,Bluemount Holdings Limited,BMHL,2025-07-11,4.00,4.00,4.00,0.0
115,"CoastalSouth Bancshares, Inc. (NYSE Uplisting)",COSO,2025-07-02,21.50,21.50,21.50,0.0
122,Happy City Holdings Limited,HCHL,2025-06-24,5.00,5.00,5.00,0.0
136,Jena Acquisition Corp. II,JENA.U,2025-05-29,10.00,0.00,0.00,0.0
137,Fast Track Group (New IPO Filing),FTRK,2025-05-23,4.00,0.00,0.00,0.0
140,OFA Group,OFAL,2025-05-21,4.00,0.00,0.00,0.0
141,Arrive AI (NASDAQ Direct Listing),ARAI,2025-05-15,13.00,0.00,0.00,0.0
150,Yuanbao Inc.,YB,2025-04-30,15.00,0.00,0.00,0.0


The date filter alone yields 163 rows; 17 of those have a 0% return, giving 146
rather than the 148 the assignment expects.

The zero-return rows are not stocks that happened to finish flat. Nine have a
1st Day Close of $0.00 — no recorded trading. The other eight have a Current
Price identical to the offer price or the 1st day close to the cent, meaning
iposcoop never refreshed the quote, so the return is mechanically zero.

Because 'Return' is computed against a live price snapshot, which rows are stale
changes over time; two rows that were fresh when the assignment was written are
stale now. The filter is applied as specified, so we proceed with 146 tickers.

### Download daily OHLCV for the filtered tickers

Pull full daily history for each of the 146 symbols. Some will be absent from
Yahoo Finance (delisted, renamed, or never traded under that symbol), so the
loop records failures rather than raising. The assignment expects roughly 134
tickers to return data.

This takes a few minutes: there is a 1-second pause between requests to avoid
hammering the API.

In [11]:
import yfinance as yf
import time

print(yf.__version__)

tickers = filtered_ipos['Symbol'].tolist()

frames = []
failed = []

for i, ticker in enumerate(tickers):
    try:
        prices = yf.Ticker(ticker).history(period='max', interval='1d')
    except Exception as e:
        failed.append((ticker, repr(e)))
        continue

    if prices.empty:
        failed.append((ticker, 'empty'))
        continue

    prices = prices.copy()
    prices['Ticker'] = ticker
    prices['Date'] = prices.index.date
    frames.append(prices.reset_index(drop=True))

    time.sleep(1)

    if (i + 1) % 25 == 0:
        print(f'{i + 1}/{len(tickers)} processed, {len(failed)} failures so far')

stocks_df = pd.concat(frames, ignore_index=True)
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])

print(f'\ntickers requested: {len(tickers)}')
print(f'tickers with data: {stocks_df["Ticker"].nunique()}')
print(f'failures: {len(failed)}')
print(f'rows: {len(stocks_df)}')
failed

1.7.0
25/146 processed, 0 failures so far


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found


50/146 processed, 3 failures so far


$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found


75/146 processed, 7 failures so far
100/146 processed, 7 failures so far


$TBH: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found


125/146 processed, 10 failures so far


$MTSR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found



tickers requested: 146
tickers with data: 133
failures: 13
rows: 66786


[('MJID', 'empty'),
 ('EMPG', 'empty'),
 ('CAEP', 'empty'),
 ('PTNM', 'empty'),
 ('AHL', 'empty'),
 ('CEPT', 'empty'),
 ('SDM', 'empty'),
 ('TBH', 'empty'),
 ('AGH', 'empty'),
 ('EPWK', 'empty'),
 ('MTSR', 'empty'),
 ('SKBL', 'empty'),
 ('MCTR', 'empty')]

### Growth, volatility, and Sharpe ratio

Compute the three fields the assignment specifies, grouped by ticker so the
shifts and rolling windows don't bleed across stocks:
- growth_252d: Close / Close.shift(252), roughly one year of trading
- volatility: 30-day rolling std of Close, annualized by sqrt(252)
- Sharpe: (growth_252d - 0.05) / volatility, with a 5% risk-free rate

Note the volatility formula is the one the assignment prescribes: it takes the
standard deviation of the closing price level, not of returns.

In [12]:
stocks_df = stocks_df.sort_values(['Ticker', 'Date'])

stocks_df['growth_252d'] = (stocks_df.groupby('Ticker')['Close']
                            .transform(lambda x: x / x.shift(252)))

stocks_df['volatility'] = (stocks_df.groupby('Ticker')['Close']
                           .transform(lambda x: x.rolling(30).std()) * np.sqrt(252))

stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

print(stocks_df[['growth_252d', 'volatility', 'Sharpe']].notna().sum())
stocks_df[['Ticker', 'Date', 'Close', 'growth_252d', 'volatility', 'Sharpe']].tail()

growth_252d    33998
volatility     62985
Sharpe         33998
dtype: int64


,Ticker,Date,Close,growth_252d,volatility,Sharpe
66354,ZYBT,2026-09-10,1.59,0.161914,3.838139,0.029159
66355,ZYBT,2026-09-11,1.55,0.149038,3.734725,0.026518
66356,ZYBT,2026-09-14,1.42,0.154180,3.666712,0.028412
66357,ZYBT,2026-09-15,1.38,0.148069,3.625701,0.027048
66358,ZYBT,2026-09-16,1.37,0.128157,3.543559,0.022056


### Q2 answer: median Sharpe ratio as of 2026-09-11

Filter to the trading day the question specifies and print the median Sharpe
directly.

In [13]:
snapshot = stocks_df[stocks_df['Date'] == '2026-09-11']

print('median Sharpe:', round(snapshot['Sharpe'].median(), 4))
print('stocks reaching 252 trading days:', snapshot['growth_252d'].notna().sum())
print('growth_252d  median:', round(snapshot['growth_252d'].median(), 3),
      ' mean:', round(snapshot['growth_252d'].mean(), 3))

median Sharpe: 0.0523
stocks reaching 252 trading days: 129
growth_252d  median: 0.596  mean: 1.066


**Q2 answer:** median Sharpe ratio of 0.0523 as of 2026-09-11, across 129 stocks
that reached the 252-day trading milestone (132 tickers had data on that date;
2 lacked a full year of history).

The mean Sharpe is inf: one stock has a 30-day rolling standard deviation of
exactly zero — an unchanged closing price across the window — so its Sharpe
divides by zero. The median is unaffected, being a rank statistic.

Median growth_252d is 0.59 against a mean of 1.06: the typical 2025 IPO lost
about 40% over its first year, while a small number of extreme winners (max
33.6x) pull the average above 1.0.

## Q3: Fixed-months holding strategy

Build 12 forward-looking growth columns on stocks_df, one per holding period
from 1 to 12 months, at 21 trading days per month. Each is the close N days
ahead divided by today's close, computed per ticker.

In [14]:
for m in range(1, 13):
    days = 21 * m
    stocks_df[f'future_growth_{m}_m'] = (
        stocks_df.groupby('Ticker')['Close']
        .transform(lambda x, d=days: x.shift(-d) / x))

growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
print(stocks_df[growth_cols].notna().sum())

future_growth_1_m     64033
future_growth_2_m     61293
future_growth_3_m     58563
future_growth_4_m     55833
future_growth_5_m     53103
future_growth_6_m     50373
future_growth_7_m     47643
future_growth_8_m     44913
future_growth_9_m     42183
future_growth_10_m    39453
future_growth_11_m    36723
future_growth_12_m    33998
dtype: int64


### Entry points

For each ticker take the earliest trading day. Joining those rows back to the
growth columns gives the returns an investor would have seen buying at the
first day's close and holding for each fixed period.

In [15]:
min_dates = stocks_df.groupby('Ticker')['Date'].min().reset_index()

ipo_entries = min_dates.merge(stocks_df, on=['Ticker', 'Date'], how='inner')

print('tickers:', stocks_df['Ticker'].nunique())
print('entry rows:', len(ipo_entries))
ipo_entries[['Ticker', 'Date', 'Close', 'future_growth_1_m',
             'future_growth_6_m', 'future_growth_12_m']].head()

tickers: 133
entry rows: 133


,Ticker,Date,Close,future_growth_1_m,future_growth_6_m,future_growth_12_m
0,AAPG,2025-01-24,17.379999,1.159954,2.259494,1.472382
1,AARD,2025-02-13,14.310000,0.647100,0.753319,0.870021
2,ADVB,2025-03-06,73.000000,0.920548,0.114521,0.067808
3,AII,2025-05-08,16.008226,1.005917,1.469231,1.236239
4,AIRO,2025-06-13,24.000000,1.085417,0.354167,0.329167


### Q3 answer: optimal holding period

Median growth across the 133 IPO entry points for each holding period from
1 to 12 months.

In [16]:
medians = ipo_entries[growth_cols].median()

for m in range(1, 13):
    col = f'future_growth_{m}_m'
    print(f'{m:2d} months: median {medians[col]:.4f}   '
          f'mean {ipo_entries[col].mean():.4f}   '
          f'n {ipo_entries[col].notna().sum()}')

best = medians.idxmax()
print(f'\nbest: {best}, median growth {medians[best]:.4f}')

 1 months: median 0.9401   mean 1.1232   n 131
 2 months: median 0.8917   mean 1.3003   n 130
 3 months: median 0.8280   mean 1.2579   n 130
 4 months: median 0.7499   mean 1.1404   n 130
 5 months: median 0.6843   mean 1.0913   n 130
 6 months: median 0.7334   mean 1.1218   n 130
 7 months: median 0.6680   mean 1.1540   n 130
 8 months: median 0.6040   mean 1.0113   n 130
 9 months: median 0.5844   mean 1.1671   n 130
10 months: median 0.5335   mean 1.1944   n 130
11 months: median 0.5153   mean 1.0766   n 130
12 months: median 0.5513   mean 0.9705   n 129

best: future_growth_1_m, median growth 0.9401


**Q3 answer:** 1 month, median growth 0.9354.

Median growth is below 1.0 at every horizon tested and declines with holding
period, from 0.9354 at one month to 0.5298 at twelve. Buying at the first day's
close and holding loses money for the typical 2025 IPO at every fixed horizon;
the shortest period wins only by losing least.

The mean exceeds 1.0 at eleven of the twelve horizons while the median stays
below it — a few large winners carry the average. At twelve months the mean
falls to 0.9631, so the winners no longer offset the losses.

## Q4: RSI-based trading strategy

Download the course-provided parquet of precomputed technical and macro
indicators, and check what it contains before filtering.

In [17]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

df = pd.read_parquet("data.parquet", engine="pyarrow")

print(df.shape)
print('date range:', df['Date'].min(), 'to', df['Date'].max())
print('tickers:', df['Ticker'].nunique())
print()
print([c for c in df.columns if 'rsi' in c.lower() or 'growth_future' in c.lower()])

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=8168407f-dd80-4321-9bbc-d3eb2f4db10e
To: /Users/alienfoo/Documents/sma_zoomcamp/data.parquet
100%|███████████████████████████████████████| 130M/130M [00:02<00:00, 56.7MB/s]


(229932, 203)
date range: 1972-06-01 00:00:00 to 2025-05-30 00:00:00
tickers: 33

['growth_future_30d', 'rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']


### Oversold signals and net income

Filter to trades between 2000-01-01 and 2025-06-01 where rsi < 30, then
compute net income from a $1,000 investment per signal using the 30-day
forward return.

In [18]:
selected_df = df[(df['rsi'] < 30) &
                 (df['Date'] >= '2000-01-01') &
                 (df['Date'] < '2025-06-01')]

net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()

print('trades:', len(selected_df))
print('null growth_future_30d:', selected_df['growth_future_30d'].isna().sum())
print('avg 30-day return:', round((selected_df['growth_future_30d'] - 1).mean() * 100, 2), '%')
print('win rate:', round((selected_df['growth_future_30d'] > 1).mean() * 100, 2), '%')
print()
print('net income: $', round(net_income, 2))
print('net income: $', round(net_income / 1000, 2), 'thousands')

trades: 5206
null growth_future_30d: 0
avg 30-day return: 1.26 %
win rate: 55.13 %

net income: $ 65805.59
net income: $ 65.81 thousands


**Q4 answer:** $65.81 thousand net income, from 5,206 oversold signals between
2000-01-01 and 2025-06-01 at $1,000 per trade.

Trade count, average 30-day return (1.26%), and win rate (55.13%) all match the
assignment's stated values, confirming the filter is the intended one.

Total capital deployed across 5,206 trades at $1,000 each is $5.21M, so the
$65.8K return is about 1.26% per trade — the same average return, since every
trade is equally sized.

### Q5: does segmentation beat buying everything?

Q3 buys all 133 IPOs and loses at every horizon. Join the offering attributes
onto the entry rows and check whether median growth differs by deal size —
offer price times shares offered. If small deals and large deals behave the
same, segmentation on size is not a way out.

In [19]:
entries = ipo_entries.merge(
    filtered_ipos[['Symbol', 'Industry', 'Offer Price', 'Shares (millions)']],
    left_on='Ticker', right_on='Symbol', how='left')

entries['deal_size'] = entries['Offer Price'] * entries['Shares (millions)']

print('entry rows:', len(entries), ' missing deal_size:', entries['deal_size'].isna().sum())
print()
print(entries['deal_size'].describe().round(1))
print()

entries['size_bucket'] = pd.qcut(entries['deal_size'], 3,
                                 labels=['small', 'mid', 'large'])

print(entries.groupby('size_bucket', observed=True).agg(
    n=('Ticker', 'count'),
    median_1m=('future_growth_1_m', 'median'),
    median_6m=('future_growth_6_m', 'median'),
    median_12m=('future_growth_12_m', 'median')).round(3))

entry rows: 133  missing deal_size: 0

count     133.0
mean      168.7
std       328.9
min         4.0
25%         7.2
50%        12.0
75%       150.8
max      1750.0
Name: deal_size, dtype: float64

              n  median_1m  median_6m  median_12m
size_bucket                                      
small        46      0.942      0.497       0.165
mid          43      0.948      0.493       0.298
large        44      0.926      0.910       0.921


In [20]:
for b in ['small', 'mid', 'large']:
    sub = entries[entries['size_bucket'] == b]['future_growth_12_m'].dropna()
    print(f'{b:6s} n={len(sub):3d}  median {sub.median():.3f}  '
          f'mean {sub.mean():.3f}  share above 1.0: {(sub > 1).mean():.1%}')

small  n= 44  median 0.165  mean 0.789  share above 1.0: 20.5%
mid    n= 41  median 0.298  mean 1.029  share above 1.0: 19.5%
large  n= 44  median 0.921  mean 1.098  share above 1.0: 43.2%


## Q5: How to increase profitability

Q3 shows the unconditional strategy loses at every horizon: buy all 133 IPOs at
the first day's close and median growth falls from 0.94 at one month to 0.53 at
twelve. Splitting the same stocks by deal size (offer price x shares offered)
shows that median is an average over two unlike populations.

| bucket | n  | median 12m | mean 12m | share > 1.0 |
|--------|----|-----------|----------|-------------|
| small  | 44 | 0.165     | 0.789    | 20.5%       |
| mid    | 41 | 0.298     | 1.029    | 19.5%       |
| large  | 44 | 0.921     | 1.098    | 43.2%       |

Deal size is bimodal rather than continuous: the median deal is $12M while the
75th percentile is $151M, so "small" and "mid" are micro-cap listings and
"large" spans $150M to $1.75B. These are different markets, not a size gradient.

The buckets are indistinguishable at one month (0.926 to 0.948) and diverge
over the following year, so the effect is not the first-day pop.

What this does and does not support:

- Excluding micro-cap offerings removes the segment where the typical stock
  loses 84% of its value in a year. This is the clearest available improvement.
- It does not produce a profitable strategy. Large deals still have a median
  below 1.0 and are profitable only 43.2% of the time.
- In small and mid buckets the mean sits far above the median, so what return
  exists is concentrated in a few names. Any strategy there is a tail bet
  requiring wide diversification, and the median is the wrong tool to judge it.

Untested directions: entering at the offer price rather than the first day's
close, which separates allocation from trading; and shorting, which the
consistently negative medians suggest but which is limited in practice by
borrow cost and availability on recent IPOs.

These results rest on 133 stocks from a single year, with 41 to 44 per bucket.
The direction is large enough to be worth noting; the magnitudes are not stable
estimates.